Imports and Feature Extraction

In [7]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine

Audio Feature Extraction

In [8]:
def analyze_audio_extended(path, bins=30):
    y, sr = librosa.load(path, sr=None)
    S = np.abs(librosa.stft(y))

    low_energy = S[:bins, :].mean()
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr).mean()
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr).mean()
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr).mean()
    zcr = librosa.feature.zero_crossing_rate(y).mean()

    return {
        "low_energy": low_energy,
        "centroid": centroid,
        "bandwidth": bandwidth,
        "rolloff": rolloff,
        "zcr": zcr
    }

def extract_feature_vector(path):
    feat = analyze_audio_extended(path)
    return np.array([feat[k] for k in ["low_energy", "centroid", "bandwidth", "rolloff", "zcr"]])

Feature → Search Query Generator

In [9]:
def generate_queries_from_features(feat_dict):
    queries = []

    if feat_dict["low_energy"] > 0.02:
        queries.append("deep bass groove music")
    else:
        queries.append("soft low-end soul")

    if feat_dict["centroid"] < 2000:
        queries.append("warm funk instrumental")
    else:
        queries.append("bright funky pop bass")

    if feat_dict["zcr"] > 0.1:
        queries.append("punchy slap bass tracks")

    if feat_dict["bandwidth"] > 3000:
        queries.append("aggressive bassline disco")

    if feat_dict["rolloff"] < 3000:
        queries.append("vintage mellow groove")

    return list(set(queries))

YouTube Search & Download

In [10]:
def download_youtube_results(keyword, save_dir="data/candidates", max_results=3):
    os.makedirs(save_dir, exist_ok=True)
    cmd = f'yt-dlp "ytsearch{max_results}:{keyword}" --extract-audio --audio-format wav -o "{save_dir}/%(title)s.%(ext)s"'
    print(f"Downloading: {keyword}")
    os.system(cmd)

Similarity Calculation

In [11]:
def compare_to_reference(reference_path, candidate_dir):
    ref_vec = extract_feature_vector(reference_path)
    results = []

    for fname in os.listdir(candidate_dir):
        if fname.endswith(".wav"):
            c_path = os.path.join(candidate_dir, fname)
            try:
                c_vec = extract_feature_vector(c_path)
                sim = 1 - cosine(ref_vec, c_vec)
                results.append((fname, sim))
            except Exception as e:
                print(f"❌ Skipping {fname} — {e}")

    return sorted(results, key=lambda x: -x[1])

Pipeline

In [12]:
def recommend_from_reference(reference_path, candidate_dir="data/candidates", top_n=5):
    # Step 1: Feature extraction from reference
    print(f"🔍 Analyzing reference track: {reference_path}")
    feat = analyze_audio_extended(reference_path)
    queries = generate_queries_from_features(feat)

    # Step 2: Clean candidate folder
    os.makedirs(candidate_dir, exist_ok=True)
    for f in os.listdir(candidate_dir):
        if f.endswith(".wav"):
            os.remove(os.path.join(candidate_dir, f))

    # Step 3: Run YouTube searches
    for query in queries:
        download_youtube_results(query, save_dir=candidate_dir, max_results=2)

    # Step 4: Compare similarity
    print("\n🎧 Comparing candidates...")
    results = compare_to_reference(reference_path, candidate_dir)

    # Step 5: Output top recommendations
    print(f"\n🎯 Top {top_n} songs similar to: {os.path.basename(reference_path)}\n")
    for fname, score in results[:top_n]:
        print(f"{fname:55} | similarity: {score:.4f}")

Run

In [13]:
reference_path = "./data/audio_raw/Billie_Jean.wav"
recommend_from_reference(reference_path, top_n=5)

🔍 Analyzing reference track: ./data/audio_raw/Billie_Jean.wav
Downloading: bright funky pop bass
Downloading: aggressive bassline disco
Downloading: deep bass groove music

🎧 Comparing candidates...

🎯 Top 5 songs similar to: Billie_Jean.wav

Funk Bass Backing Track in G Minor.wav                  | similarity: 1.0000
‌House Rally Playlist.wav                               | similarity: 0.9994
Funk No.1 - TOKYO GROOVE JYOSHI (feat. Juna Serita & Harumo Imai).wav | similarity: 0.9974
How to Get the PERFECT Bass Tone.wav                    | similarity: 0.9969
Black Cats Jazz Night： Retro Grooves & Deep Bass.wav    | similarity: 0.9949
